# 原39项直接判断SOXL是否胜现金：固定二分类信息门

一次训练期Logistic拟合，原f696配置意图与训练先验为预定对照；复用2023–2026历史只作开发反馈。过信息门才另冻经济动作。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json,shlex,tempfile
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score,brier_score_loss
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence,_ssh
root=Path.cwd();card=root/'research/stage1_original39_soxl_cash_binary_information_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
assert sha(root/s['dataset_card']/'summary.json')==s['dataset_summary_sha256']
assert sha(root/s['f696_card']/'summary.json')==s['f696_summary_sha256']
assert sha(root/s['prior_pairwise_rank_result'])==s['prior_pairwise_rank_result_sha256']
assert sha(root/s['provider']/'qlib_store_manifest.json')==s['provider_manifest_sha256']
a=root/s['dataset_card'];f=root/s['f696_card'];sa=json.loads((a/'summary.json').read_text());sf=json.loads((f/'summary.json').read_text())
assert sa['nas_readback']['verified'] and sa['nas_readback']['sha256']==s['dataset_NAS_sha256']
rb=json.loads((root/s['f696_readback']).read_text())
assert rb['status']=='passed' and rb['all_artifact_hashes_match'] and rb['archive_sha256']==sf['nas_evidence']['sha256']==s['f696_NAS_sha256']
assert sha(a/'qlib_recorder'/s['dataset_experiment_id']/s['dataset_recorder_id']/'artifacts/inputs/dataset')==s['dataset_artifact_sha256']
assert sha(f/'qlib_recorder'/s['f696_experiment_id']/s['f696_recorder_id']/'artifacts/inputs/frame')==s['f696_frame_sha256']
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['dataset_experiment_id'],recorder_id=s['dataset_recorder_id']);dataset=rec.load_object('inputs/dataset')
with R.uri_context(str((f/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['f696_experiment_id'],recorder_id=s['f696_recorder_id']);frame=rec.load_object('inputs/frame').sort_values('trade_date').reset_index(drop=True)
tr_full=dataset.prepare('train',col_set=['feature','label','meta']);he_full=dataset.prepare('held',col_set=['feature','label','meta'])
assert len(he_full)==len(frame)==944
tr=tr_full.loc[tr_full[('meta','train_mature')].astype(bool)];he=he_full.loc[he_full[('meta','held_mature')].astype(bool)]
assert len(tr)==3014 and len(he)==940
features=list(tr['feature'].columns);assert len(features)==39 and list(he['feature'].columns)==features
train_dates=pd.DatetimeIndex(tr.index.get_level_values('datetime'));held_dates=pd.DatetimeIndex(he.index.get_level_values('datetime'))
full_held_dates=pd.DatetimeIndex(he_full.index.get_level_values('datetime'));frame_dates=pd.DatetimeIndex(frame.signal_date)
assert train_dates.is_unique and held_dates.is_unique and full_held_dates.equals(frame_dates) and held_dates.equals(frame_dates[:940])
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'))
assert train_dates.isin(calendar).all() and full_held_dates.isin(calendar).all()
assert train_dates.max()<held_dates.min()
last_train_available=pd.to_datetime(tr[('meta','label_available_at')],utc=True).max()
assert last_train_available < pd.Timestamp(held_dates.min()).tz_localize('UTC')
train_margin=tr[('label','SOXL_relative_H5')].to_numpy(float)-tr[('label','cash_advantage_H5')].to_numpy(float)
held_margin=he[('label','SOXL_relative_H5')].to_numpy(float)-he[('label','cash_advantage_H5')].to_numpy(float)
assert np.isfinite(train_margin).all() and np.isfinite(held_margin).all()
train_ties=np.abs(train_margin)<=1e-12;held_ties=np.abs(held_margin)<=1e-12
train_y=(train_margin[~train_ties]>0).astype(int);held_y=(held_margin[~held_ties]>0).astype(int)
assert set(train_y)==set(held_y)=={0,1} and int(held_ties.sum())==0
train_x=tr['feature'][features].to_numpy(float)[~train_ties];held_x=he['feature'][features].to_numpy(float)[~held_ties]
assert not np.isinf(train_x).any() and not np.isinf(held_x).any()
assert np.isfinite(train_x).any(axis=0).all() and np.isfinite(held_x).any(axis=0).all()
control=(frame.soxl_weight-frame.cash_weight).to_numpy(float)[:940][~held_ties]
prior=json.loads((root/s['prior_pairwise_rank_result']).read_text());control_auc=float(roc_auc_score(held_y,control))
assert abs(control_auc-prior['held_full']['auc'])<1e-12
state=card/'run_state.json';state.write_text(json.dumps({'status':'fit_attempted','plan_sha256':sha(plan_path),'new_fit_attempts':1,'new_fits_completed':0},indent=2)+'\n')
model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(C=1.0,solver='lbfgs',max_iter=2000,random_state=71))
model.fit(train_x,train_y)
state.write_text(json.dumps({'status':'fit_completed','plan_sha256':sha(plan_path),'new_fit_attempts':1,'new_fits_completed':1},indent=2)+'\n')
prob=model.predict_proba(held_x)[:,1]
assert np.isfinite(prob).all() and (prob>0).all() and (prob<1).all()
train_prior=float(train_y.mean())
def metrics(mask):
 y=held_y[mask];p=prob[mask];c=control[mask]
 return {'rows':len(y),'positive':int(y.sum()),'auc':float(roc_auc_score(y,p)) if len(set(y))==2 else None,
         'control_auc':float(roc_auc_score(y,c)) if len(set(y))==2 else None,
         'brier':float(brier_score_loss(y,p)),'prior_brier':float(brier_score_loss(y,np.full(len(y),train_prior))),
         'prob_min_max':[float(p.min()),float(p.max())]}
full=metrics(np.ones(len(held_y),bool));annual=[];quarter=[]
years=held_dates.year;quarters=held_dates.quarter
for year in sorted(set(years)):
 annual.append({'year':int(year),'partial_endpoint':year in (2023,2026),**metrics(years==year)})
for year in sorted(set(years)):
 for q in (1,2,3,4):
  mask=(years==year)&(quarters==q)
  if mask.any():quarter.append({'year':int(year),'quarter':q,**metrics(mask)})
halves=[{'start':'2023-01-03','end':'2024-12-31',**metrics(np.arange(len(held_y))<502)},
        {'start':'2025-01-02','end':'2026-10-07',**metrics(np.arange(len(held_y))>=502)}]
assert len(annual)==4 and len(quarter)==16 and halves[0]['rows']==502 and halves[1]['rows']==438
rng=np.random.default_rng(20261024);n=944;reps=np.empty(10000,float);valid=0
for i in range(10000):
 indices=[]
 while len(indices)<n:
  start=int(rng.integers(0,n));length=int(rng.geometric(1/63));indices.extend((start+np.arange(length))%n)
 idx=np.asarray(indices[:n],int);idx=idx[idx<940]
 y=held_y[idx]
 if len(set(y))<2:reps[i]=np.nan;continue
 reps[i]=float(roc_auc_score(y,prob[idx])-roc_auc_score(y,control[idx]));valid+=1
assert valid>=9900
lower=float(np.nanquantile(reps,.025));upper=float(np.nanquantile(reps,.975))
bootstrap={'draws':10000,'valid':valid,'calendar_rows':944,'scored_rows':940,'mean_auc_improvement':float(full['auc']-full['control_auc']),'lower_2p5':lower,'upper_97p5':upper}
comp_years=[z for z in annual if z['year'] in (2024,2025)];year2023=annual[0]
gates={'full_auc_above_random_and_control_plus_0p02':bool(full['auc']>.5 and full['auc']>full['control_auc']+.02),
       'partial2023_auc_above_random':bool(year2023['auc']>.5),
       'both_complete_years_auc_above_random':all(z['auc']>.5 for z in comp_years),
       'both_halves_auc_above_random':all(z['auc']>.5 for z in halves),
       'full_brier_below_train_prior':bool(full['brier']<full['prior_brier']),
       'corrected_auc_improvement_lower_positive':bool(lower>0)}
pass_gate=all(gates.values())
pred=pd.DataFrame({'signal_date':held_dates,'target_SOXL_over_cash':held_y,'candidate_probability':prob,'f696_intent_score':control})
result={'decision':'conditional_direct_pairwise_binary_information_pass_requires_economic_mapping' if pass_gate else 'not_promoted_fixed_original39_SOXL_cash_binary_information_failed',
        'plan_sha256':sha(plan_path),'train_mature_rows':len(tr),'train_ties':int(train_ties.sum()),'train_positive':int(train_y.sum()),'train_prior':train_prior,
        'held_source_rows':944,'held_mature_rows':len(he),'held_ties':int(held_ties.sum()),'last_four_unknown':4,
        'feature_count':len(features),'train_feature_nonfinite':int((~np.isfinite(train_x)).sum()),'held_feature_nonfinite':int((~np.isfinite(held_x)).sum()),
        'full':full,'annual':annual,'quarter':quarter,'halves':halves,'bootstrap':bootstrap,'gates':gates,'information_pass':pass_gate,
        'new_market_GETs':0,'new_model_fits':1,'new_bootstraps':1,'new_native_accounts':0,'new_Qlib_Recorders':1,
        'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['H5 gross open-price binary labels omit Tiered fees and own-state risk feedback','2023-2026 held market history has been reused repeatedly and is not independent validation','No prediction threshold, allocation/cap mapping or complete f696 account is tested']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();dataset.config(dump_all=True,recursive=True)
 rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset,source_recorder_ids={'labels':s['dataset_recorder_id'],'f696':s['f696_recorder_id']})
 rec.save_objects(artifact_path='models',binary_SOXL_cash=model)
 rec.save_objects(artifact_path='predictions',held=pred)
 rec.save_objects(artifact_path='analysis',result=result,annual=annual,quarter=quarter,halves=halves,bootstrap_replicates=reps)
 rec.log_metrics(train_mature_rows=len(tr),held_mature_rows=len(he),held_auc=full['auc'],f696_control_auc=full['control_auc'],held_brier=full['brier'],auc_block_lower=lower,new_model_fits=1,new_native_accounts=0)
 rid=rec.id;eid=str(rec.experiment_id)
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':sha(plan_path),'result':'result.json',
         'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'source_recorder_ids':{'labels':s['dataset_recorder_id'],'f696':s['f696_recorder_id']},
         'information_pass':pass_gate,'gates':gates,'new_model_fits':1,'new_native_accounts':0,
         'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/dataset','models/binary_SOXL_cash','predictions/held','analysis/bootstrap_replicates']}}
binding=json.loads((a/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2026-10-07')
(card/'nas_binding.json').write_text(json.dumps(binding,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
with tempfile.TemporaryDirectory(prefix='soxl-cash-binary-readback-') as temp:
 restored=Path(temp)/'card';restored.mkdir();materialize_nas_evidence(card/'nas_binding.json',card=restored)
 for art in ('models/binary_SOXL_cash','predictions/held'):
  rel=Path('qlib_recorder')/eid/rid/'artifacts'/art;assert sha(card/rel)==sha(restored/rel)
summary['nas_evidence']=evidence;summary['nas_readback']={'verified':True,'sha256':remote}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
state.write_text(json.dumps({'status':'completed','plan_sha256':sha(plan_path),'new_fit_attempts':1,'new_fits_completed':1,'recorder_id':rid,'NAS_archive_sha256':remote},indent=2)+'\n')
print({'decision':result['decision'],'full':full,'annual_auc':[(z['year'],z['auc']) for z in annual],'bootstrap':bootstrap,'gates':gates,'recorder_id':rid,'NAS_sha256':remote})


{'decision': 'not_promoted_fixed_original39_SOXL_cash_binary_information_failed', 'full': {'rows': 940, 'positive': 539, 'auc': 0.47076187083312127, 'control_auc': 0.46799513276178756, 'brier': 0.2717825414393622, 'prior_brier': 0.2447192506962501, 'prob_min_max': [0.14301071915641142, 0.8576813068801505]}, 'annual_auc': [(2023, 0.5006433350488935), (2024, 0.5423739711934156), (2025, 0.40601503759398505), (2026, 0.42863778887469517)], 'bootstrap': {'draws': 10000, 'valid': 10000, 'calendar_rows': 944, 'scored_rows': 940, 'mean_auc_improvement': 0.0027667380713337075, 'lower_2p5': -0.07290218539938847, 'upper_97p5': 0.09594465360046929}, 'gates': {'full_auc_above_random_and_control_plus_0p02': False, 'partial2023_auc_above_random': True, 'both_complete_years_auc_above_random': False, 'both_halves_auc_above_random': False, 'full_brier_below_train_prior': False, 'corrected_auc_improvement_lower_positive': False}, 'recorder_id': 'aac3f60c499b43449131b98b51f412b8', 'NAS_sha256': '6a8761dfe8

## 中文结论

# 原39项直接判断SOXL胜现金：2025后排序反转，固定信息门失败

固定原Stage1决策前39项 Qlib DatasetH，训练2011–2022的 **3014** 个成熟五日毛开盘价成对标签（SOXL胜现金1697），复用2023–2026的 **940** 个成熟评价标签（正539；完整944信号日末4未知）。一次 `median imputer→standard scaler→LogisticRegression(C=1)` 拟合，未调参、未选交易阈值；对照为精确f696事前SOXL减现金提案排序和训练阳性率常数。来源SHA、NAS限定回读及Qlib时钟通过。

全段候选AUC **0.47076**，f696对照 **0.467995**，仅增 **0.00277**；候选Brier **0.27178**，劣于训练常数 **0.24472**。部分2023与完整2024 AUC **0.50064/0.54237**，但完整2025 **0.40602**、部分2026 **0.42864**；前后固定半段 **0.52401/0.41736**。全部16季度留在 `result.json`。按完整944日历、平均块长63日、10000次固定重抽样，候选相对f696的AUC增量2.5%下界 **−0.07290**。六项预登记信息门只有2023部分年AUC略高于0.5，通过 **1/6**。

决定 `not_promoted_fixed_original39_SOXL_cash_binary_information_failed`。它确实改变了旧双Ridge幅度头的接收任务，但2025后发生方向性失效，不能凭2023/2024好段改阈值、加年份开关或把概率强行映射到f696现金/风险容量。本实验只有毛H5信息，没有分红/手续费后的完整自身状态账户；2023–2026价格历史反复复用，非独立前向。SOXX硬门、2/5bp与合格前向均未达成。

首次执行在拟合前被时区感知与非感知时间直接比较拦下，`attempt1_preflight_failure.json`保留；只把两边统一UTC并要求训练标签最后可用时刻严格早于首评价信号日后，单次拟合/单次bootstrap完成，未改冻结模型或门槛。Qlib Recorder `aac3f60c499b43449131b98b51f412b8` 保存模型、940日预测及10000次复制值；NAS归档SHA-256 `6a8761dfe8b4b223b88264d60a8041a7db2134c3360564efed1b4d5a46177569` 已定向恢复模型与预测工件。